# CBRAIN OASIS Write-up and Packaging (Step 5)

Goal: freeze a final project snapshot and ensure documentation-level consistency for transfer and review.


In [ ]:
# 1) Load all stage summaries
from pathlib import Path
import json
import pandas as pd
import textwrap

ROOT = Path.cwd()
cohort_dir = ROOT / 'results' / 'cohort'
prep_dir = ROOT / 'results' / 'preprocessing'
model_dir = ROOT / 'results' / 'modeling'
docs_dir = ROOT / 'docs'

cohort_summary = json.loads((cohort_dir / 'cohort_summary.json').read_text())
split_payload = json.loads((cohort_dir / 'split_subjects.json').read_text())
prep_summary = json.loads((prep_dir / 'preprocessing_summary.json').read_text())
baseline_metrics = json.loads((model_dir / 'baseline_metrics.json').read_text())
rf_metrics = json.loads((model_dir / 'step3b_random_forest_metrics.json').read_text())
step3c_summary = json.loads((model_dir / 'step3c_calibration_summary.json').read_text())
step3d_summary = json.loads((model_dir / 'step3d_error_group_summary.json').read_text())
step4_summary = json.loads((model_dir / 'step4_interpretation_summary.json').read_text())

print('Loaded stage summaries successfully.')


In [ ]:
# 2) Build and write final project snapshot JSON
snapshot = {
    'project': 'cbrain_oasis_cognition',
    'snapshot_date': '2026-03-23',
    'pipeline_status': 'complete_through_step5',
    'cohort': {
        'raw_row_count': cohort_summary['raw_row_count'],
        'baseline_row_count': cohort_summary['baseline_row_count'],
        'raw_subject_count': cohort_summary['raw_subject_count'],
        'label_positive_rate': cohort_summary['label_positive_rate'],
        'train_subject_count': split_payload['train_subject_count'],
        'validation_subject_count': split_payload['validation_subject_count'],
        'split_overlap_count': split_payload['overlap_count'],
    },
    'preprocessing': {
        'final_feature_order': prep_summary['final_feature_order'],
        'train_missing_counts': prep_summary['missing_counts_transformed_X_train'],
        'validation_missing_counts': prep_summary['missing_counts_transformed_X_validation'],
    },
    'models': {
        'baseline_logistic_validation': baseline_metrics['validation_metrics'],
        'random_forest_validation': rf_metrics['validation_metrics'],
        'comparison': {
            'roc_auc_delta_rf_minus_logistic': rf_metrics['validation_metrics']['roc_auc'] - baseline_metrics['validation_metrics']['roc_auc'],
            'pr_auc_delta_rf_minus_logistic': rf_metrics['validation_metrics']['pr_auc'] - baseline_metrics['validation_metrics']['pr_auc'],
            'f1_delta_rf_minus_logistic': rf_metrics['validation_metrics']['f1'] - baseline_metrics['validation_metrics']['f1'],
        },
    },
    'threshold_and_calibration': step3c_summary,
    'error_analysis': step3d_summary,
    'interpretation': {
        'top_logistic_abs_coef_predictors': step4_summary['top_logistic_abs_coef_predictors'],
        'top_random_forest_predictors': step4_summary['top_random_forest_predictors'],
        'causal_guardrail': step4_summary['causal_guardrail'],
    },
}

snapshot_path = model_dir / 'step5_project_snapshot.json'
snapshot_path.write_text(json.dumps(snapshot, indent=2))
print('Wrote', snapshot_path)
snapshot


In [ ]:
# 3) Write Step 5 delivery summary doc
val_log = baseline_metrics['validation_metrics']
val_rf = rf_metrics['validation_metrics']

summary_doc = f"""# Step 5: Final Write-up and Transfer Packaging

Date: March 23, 2026 (IST)

## What this step does

Step 5 finalizes the project packet for reproducible review and downstream handoff.
It does not retrain models; it consolidates frozen artifacts from Steps 1-4.

## Frozen Status

- Cohort and leakage contracts are locked from Step 2A.
- Preprocessing is train-fit-only and frozen from Step 2B.
- Baseline logistic and RF comparison metrics are fixed from Step 3A and 3B.
- Threshold, calibration, and error analyses are fixed from Step 3C and 3D.
- Interpretation guardrails are fixed from Step 4.

## Final Validation Snapshot

- Logistic ROC-AUC: {val_log['roc_auc']:.4f}
- Logistic PR-AUC: {val_log['pr_auc']:.4f}
- RF ROC-AUC: {val_rf['roc_auc']:.4f}
- RF PR-AUC: {val_rf['pr_auc']:.4f}
- Subject overlap (train vs validation): {split_payload['overlap_count']}

## Transfer Artifacts

- results/modeling/step5_project_snapshot.json
- docs/project_brief.md
- docs/application_mapping.md
- docs/interpretation_step4.md

## Immediate Next Work

1. Replace single split with nested CV or repeated subject-level resampling.
2. Lock operating threshold on untouched holdout cohort.
3. Expand uncertainty reporting (bootstrap CIs for AUC, F1, calibration).
"""

step5_doc = docs_dir / 'step5_delivery_summary.md'
step5_doc.write_text(summary_doc.strip() + "\n")
print('Wrote', step5_doc)


In [ ]:
# 4) Consistency checks
assert split_payload['overlap_count'] == 0, 'Subject overlap must remain zero'
assert set(pd.read_csv(ROOT / 'results/cohort/baseline_model_table.csv')['cognitive_impairment'].unique()).issubset({0, 1})
assert (model_dir / 'step5_project_snapshot.json').exists()
assert (docs_dir / 'step5_delivery_summary.md').exists()
print('Step 5 checks passed.')
